# Chapter 13: RAG for Enterprise Knowledge

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build a **Knowledge Base Assistant** using Retrieval-Augmented Generation (RAG) that answers BA/QA questions from your organization's documentation -- requirements, test plans, process docs, and standards.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Basic understanding of embeddings and vector search


In [ ]:
# Setup
!pip install -q openai python-dotenv numpy

import os
import json
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'
EMBED_MODEL = 'text-embedding-3-small'

## 1. Build the Knowledge Base

Create a simple in-memory vector store with enterprise BA/QA documents.


In [ ]:
# Simulated enterprise knowledge base documents
documents = [
    {
        'id': 'DOC-001',
        'title': 'Requirements Writing Standard',
        'category': 'Standards',
        'content': 'All requirements must follow the EARS (Easy Approach to Requirements Syntax) pattern. Ubiquitous requirements use "The [system] shall [action]". Event-driven use "When [trigger], the [system] shall [action]". State-driven use "While [state], the [system] shall [action]". Each requirement must have a unique ID, be testable, and be traced to a business objective.'
    },
    {
        'id': 'DOC-002',
        'title': 'Test Case Design Guidelines',
        'category': 'QA Process',
        'content': 'Test cases must cover: equivalence partitioning, boundary value analysis, decision table testing, and state transition testing. Each test case requires: unique ID, preconditions, steps, expected results, and priority. Critical path tests must be automated. Regression suite should not exceed 4 hours execution time.'
    },
    {
        'id': 'DOC-003',
        'title': 'Agile Definition of Done',
        'category': 'Process',
        'content': 'A story is done when: code is peer-reviewed, unit tests pass (>80% coverage), integration tests pass, acceptance criteria verified, documentation updated, no critical or high defects open, performance benchmarks met, security scan clean, and product owner accepts the demo.'
    },
    {
        'id': 'DOC-004',
        'title': 'Defect Severity Classification',
        'category': 'QA Process',
        'content': 'Critical: System crash, data loss, security breach. SLA: fix within 4 hours. High: Major feature broken, no workaround. SLA: fix within 24 hours. Medium: Feature partially broken, workaround exists. SLA: fix within 1 sprint. Low: Cosmetic issue, minor UX problem. SLA: fix when capacity allows.'
    },
    {
        'id': 'DOC-005',
        'title': 'API Testing Standards',
        'category': 'QA Process',
        'content': 'All REST APIs must be tested for: correct HTTP status codes (200, 201, 400, 401, 403, 404, 500), request/response schema validation, authentication and authorization, rate limiting, pagination, error message format consistency, and performance (p95 < 500ms for reads, p95 < 2s for writes).'
    },
    {
        'id': 'DOC-006',
        'title': 'Business Requirements Document Template',
        'category': 'Standards',
        'content': 'BRD must include: Executive Summary, Business Objectives, Scope (in/out), Stakeholder Analysis, Current State Analysis, Future State Description, Functional Requirements, Non-Functional Requirements, Assumptions, Constraints, Dependencies, Risks, Success Metrics, and Approval Sign-off section.'
    }
]

# Generate embeddings for all documents
def get_embedding(text: str) -> list:
    response = client.embeddings.create(model=EMBED_MODEL, input=text)
    return response.data[0].embedding

# Build the vector store
for doc in documents:
    doc['embedding'] = get_embedding(f"{doc['title']}: {doc['content']}")

print(f'Knowledge base loaded: {len(documents)} documents')
print(f'Embedding dimension: {len(documents[0]["embedding"])}')

## 2. Semantic Search and Retrieval

Implement vector similarity search to find relevant documents for a query.


In [ ]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def search_knowledge_base(query: str, top_k: int = 3) -> list:
    """Search the knowledge base using semantic similarity."""
    query_embedding = get_embedding(query)
    scored = []
    for doc in documents:
        score = cosine_similarity(query_embedding, doc['embedding'])
        scored.append({**doc, 'similarity': round(float(score), 4)})
    scored.sort(key=lambda x: x['similarity'], reverse=True)
    return scored[:top_k]

# Test semantic search
queries = [
    'How should I write a good requirement?',
    'What makes a bug critical vs high priority?',
    'What should I include in a BRD?',
]

for query in queries:
    print(f'\nQuery: "{query}"')
    results = search_knowledge_base(query, top_k=2)
    for r in results:
        print(f'  [{r["similarity"]}] {r["id"]}: {r["title"]}')

In [ ]:
def ask_knowledge_base(question: str) -> str:
    """RAG: retrieve relevant docs, then generate an answer grounded in them."""
    # Retrieve
    relevant_docs = search_knowledge_base(question, top_k=3)
    context = '\n\n'.join([
        f"[{doc['id']}: {doc['title']}]\n{doc['content']}" 
        for doc in relevant_docs
    ])
    
    # Generate
    prompt = f"""Answer the following question using ONLY the provided context. 
If the context doesn't contain enough information, say so.
Always cite the document ID(s) you used.

Context:
{context}

Question: {question}

Answer:"""
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'You are a helpful BA/QA knowledge assistant. Answer based only on the provided documents.'},
            {'role': 'user', 'content': prompt}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content

# Test the RAG assistant
test_questions = [
    'What patterns should I use when writing requirements?',
    'How quickly should a critical defect be fixed?',
    'What testing techniques are required for test case design?',
    'What are the criteria for a story to be considered done?',
]

for q in test_questions:
    print(f'\nQ: {q}')
    print(f'A: {ask_knowledge_base(q)}')
    print('-' * 60)

## Exercises
1. Add more documents to the knowledge base and test with complex multi-document questions.
2. Implement a document chunking strategy for long documents (split into paragraphs with overlap).
3. Add metadata filtering (e.g., only search within 'QA Process' category).
4. Build a feedback loop where users can rate answers and improve retrieval.
